# Reconocimiento de actividades con sEMG: CPC–xLSTM con frecuencia configurable

**Proyecto Integrador — Equipo 1**

Este notebook entrena y compara modelos para reconocer siete actividades de manejo manual de cargas a partir de las 8 envolventes sEMG (`Norm_RMS`) que prepara y exporta `avance2.equipo1.ipynb`:

| Etiqueta | Actividad |
|---|---|
| `N` | Postura neutral (de pie, sin manipular la carga) |
| `LF` | Levantar la carga del piso |
| `K` | Sostener la carga |
| `PT` | Colocar la carga en la mesa |
| `LT` | Levantar la carga de la mesa |
| `W` | Caminar con la carga |
| `PF` | Colocar la carga en el piso |

## Idea general

El enfoque es **Contrastive Predictive Coding (CPC)** (van den Oord et al., 2018; aplicado a reconocimiento de actividades por Haresamudram, Essa y Plötz, 2020). Tiene dos fases:

1. **Preentrenamiento sin etiquetas.** Un encoder convolucional convierte cada instante de la señal en un vector `z_t`; un modelo autorregresivo (`g_ar`) resume el pasado en un contexto `c_t`; y una pérdida contrastiva (InfoNCE) le pide a `c_t` reconocer cuál es el `z_{t+k}` verdadero entre los de otras ventanas del batch. Para lograrlo, el modelo tiene que aprender la estructura temporal de la señal. Esta fase aprovecha **toda** la señal, incluidos los tramos sin etiqueta (≈ 31 % del registro).
2. **Clasificación.** Con las representaciones aprendidas (`c_t`) se clasifican las actividades, ya sea con una sonda lineal sobre el modelo congelado o ajustando todo el modelo con las etiquetas.

Como `g_ar` se comparan tres arquitecturas: la **GRU** del artículo de CPC para actividades y dos variantes de **xLSTM** (Beck et al., 2024): **sLSTM** y **mLSTM**. La implementación base de CPC está en `01_Pipeline_CPC.ipynb`.

## Preguntas que responde

1. ¿El preentrenamiento CPC mejora la clasificación frente a la **misma red sin entrenar** y frente a entrenarla **sólo con etiquetas**?
2. ¿Qué `g_ar`, qué horizonte de predicción y qué representación de la ventana funcionan mejor?
3. ¿Ayuda formar los batches de CPC con ventanas de **un solo sujeto**?
4. ¿Cuánto aportan las características manuales del Avance 2, solas o combinadas con CPC?
5. ¿El modelo reconoce la actividad por la **señal** o por el **orden fijo del protocolo**?

## Decisiones de diseño

| Decisión | Qué se hace | Por qué |
|---|---|---|
| Datos | Se leen de `notebooks/_cache/avance2/fs{TARGET_FS}/` y se aplica el escalador exportado (`clip` por cuantiles + `log1p` + `RobustScaler`) | El escalador se ajustó sólo con sujetos de entrenamiento; recalcularlo con otros sujetos filtraría información |
| Frecuencia | `TARGET_FS` configurable (12, 25 o 50 Hz); la ventana de 4 s tiene `4 · TARGET_FS` pasos | Permite comparar costo y desempeño sin cambiar el código |
| Partición | Por sujeto: entrenamiento 1–10, validación 11–12, prueba 13–14 | Mide la generalización a **personas nuevas**; validación elige, prueba sólo confirma |
| Etiqueta de la ventana | La actividad del **último instante** de la ventana | Conserva las siete clases (muchas actividades duran menos de 4 s) y plantea la tarea como "qué ocurre ahora dados los últimos 4 s" |
| Encoder | Convoluciones **causales** (relleno sólo a la izquierda) | Con relleno simétrico, cada `z_t` vería hasta 3 pasos del futuro, que es justo lo que CPC debe predecir |
| Horizontes de CPC | Se definen en **segundos** (A = 0.5 s, B = 1 s) y se convierten a pasos | Mantiene el mismo significado temporal a cualquier frecuencia |
| Ventanas de CPC | Recortes aleatorios de la serie continua en cada batch | Cada época ve ventanas distintas, con más variedad que un índice fijo |
| Negativos de InfoNCE | `mezclado` (de varios sujetos) o `mismo_sujeto` | Con negativos de otras personas, la tarea se puede resolver reconociendo a la persona en lugar de la actividad; `mismo_sujeto` elimina ese atajo |
| Representación | `c_T`, `cola` (media del último segundo), `media` y `c_T+media` | Con etiqueta en el instante final, la información más útil debería estar al final de la ventana |
| Épocas | Hasta `EPOCHS_MAX`, con parada temprana por F1 de validación | El desempeño en clasificación suele dejar de mejorar antes de que lo haga la pérdida contrastiva |
| Semillas | Barrido con una semilla; las mejores configuraciones se repiten con varias | Diferencias de ±0.02–0.03 de F1 pueden deberse sólo a la inicialización |
| Prueba | Cerrada hasta el final (`TEST_OPEN = False`) | Todas las decisiones se toman con validación |

## Flujo

1. Configuración
2. Datos exportados
3. Modelo: CPC con encoder causal y las tres variantes de `g_ar`
4. Verificaciones y estimación de tiempo
5. Herramientas de entrenamiento y evaluación
6. Etapa 1 — barrido de modelo × horizonte × modo de batch con una semilla
7. Etapa 2 — las mejores configuraciones con varias semillas
8. Diagnósticos: por clase, desde el cambio de actividad, orden del protocolo y sonda de sujeto
9. Línea base con características manuales y vista híbrida
10. Etapa 3 — ajuste fino del modelo completo
11. Comparación final en validación, total y por sujeto
12. Prueba (una sola vez)
13. Cómo leer los resultados

## Cómo ejecutarlo

1. **Prerrequisito:** ejecutar `avance2.equipo1.ipynb` con el mismo `TARGET_FS` y la llamada a `export_artifacts()` activa, para que exista `notebooks/_cache/avance2/fs{TARGET_FS}/`.
2. Elegir `TARGET_FS` en la sección 1 y ejecutar todo. En una Mac con Apple Silicon (MPS), una corrida completa tomó del orden de 9 h a 12 Hz y 13 h a 25 Hz. Conviene ejecutarlo con el equipo sin suspender (`caffeinate -i`).
3. **Corridas guardadas:** cada entrenamiento se guarda en `checkpoints/cpc06_fs{TARGET_FS}/` (pesos `.pt` e historial `.json`). Al reejecutar, las corridas existentes no se repiten (`RERUN = False`): se cargan y se muestra su resumen. Así el notebook se puede interrumpir y retomar.
4. `QUICK = True` hace una corrida de humo de pocos minutos, en una carpeta aparte, para comprobar que todo funciona.
5. Si se cambia un parámetro de entrenamiento después de haber corrido, usar `RERUN = True` o borrar la carpeta de corridas; si no, se reutilizarán corridas hechas con la configuración anterior.

**Archivos que genera:** `checkpoints/cpc06_fs{TARGET_FS}/` (no se versiona) y `notebooks/_cache/cpc06_fs{TARGET_FS}_resultados.csv` (métricas de las etapas 1 y 2, sí se versiona).

# 1. Configuración

In [ ]:
import copy, json, math, time
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# ----------------------------- Experimento -----------------------------
TARGET_FS = 12                         # 12, 25 o 50: debe existir notebooks/_cache/avance2/fs{TARGET_FS}/
QUICK = False                          # True = corrida de humo (pocas épocas y sub-muestras, carpeta aparte)
MODELS = ("GRU", "mLSTM", "sLSTM")     # en orden de costo
HORIZONS_S = {"A": 0.5, "B": 1.0}      # horizonte máximo de CPC en segundos
MAX_PRED_HEADS = 12                    # nº máximo de matrices W_k; si hay más pasos, se espacian
BATCH_MODES = ("mezclado", "mismo_sujeto")
CPC_SAMPLING = "recortes_aleatorios"   # o "indice_fijo" (sólo las ventanas de cpc_index, salto de 2 s)
CHANNEL_MODE = "8"                     # "8" o "pares4" (promedio de ch1–2, ch3–4, ch5–6, ch7–8)
SEEDS = (0, 1, 2)
TOP_K_CONFIGS = 2                      # configuraciones de la etapa 1 que se repiten con todas las semillas

# ----------------------------- Entrenamiento CPC -----------------------------
EPOCHS_MAX = 20
EVAL_EVERY = 2                         # cada cuántas épocas se evalúa la sonda lineal en validación
PATIENCE = 3                           # evaluaciones seguidas sin mejorar antes de parar
BATCH_SIZE = 128
LR = 5e-4
GRAD_CLIP = 1.0
REPRESENTATIONS = ("c_T", "cola", "media", "c_T+media")
EARLY_STOP_REPS = ("c_T", "cola")      # alineadas con la etiqueta del instante final
PROBE_TRAIN_MAX = 15_000               # ventanas de entrenamiento (estratificadas) para las sondas lineales

# ----------------------------- Ajuste fino -----------------------------
FINETUNE_EPOCHS = 15
FINETUNE_PATIENCE = 3
FINETUNE_BATCH = 256
FINETUNE_LR_HEAD, FINETUNE_LR_BACKBONE = 1e-3, 1e-4
FINETUNE_SEEDS = (0, 1, 2)
FINETUNE_TOP_K = 2                     # configuraciones de la etapa 2 que pasan a ajuste fino

# ----------------------------- Control -----------------------------
TEST_OPEN = False                      # True sólo al final, con todas las decisiones congeladas
RERUN = False                          # True = repetir corridas aunque ya existan
SEED_DATA = 0
DEVICE_OVERRIDE = None                 # p. ej. "cpu" para descartar diferencias de MPS

STEPS_CAP = VAL_CAP = FINETUNE_TRAIN_CAP = None
if QUICK:
    EPOCHS_MAX, EVAL_EVERY, PATIENCE, SEEDS, TOP_K_CONFIGS = 2, 1, 1, (0, 1), 1
    PROBE_TRAIN_MAX, STEPS_CAP, VAL_CAP = 2_000, 4, 2_000
    FINETUNE_EPOCHS, FINETUNE_TRAIN_CAP, FINETUNE_SEEDS, FINETUNE_TOP_K = 1, 2_000, (0,), 1
# ---------------------------------------------------------------------------

if DEVICE_OVERRIDE:
    DEVICE = torch.device(DEVICE_OVERRIDE)
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")

REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
EXPORT_DIR = REPO_DIR / "notebooks" / "_cache" / "avance2" / f"fs{TARGET_FS}"
TAG = f"fs{TARGET_FS}" + ("_quick" if QUICK else "") + ("" if CHANNEL_MODE == "8" else f"_{CHANNEL_MODE}")
RUN_DIR = REPO_DIR / "checkpoints" / f"cpc06_{TAG}"
SUMMARY_CSV = REPO_DIR / "notebooks" / "_cache" / f"cpc06_{TAG}_resultados.csv"
for sub in ("cpc", "sin_entrenar", "finetune"):
    (RUN_DIR / sub).mkdir(parents=True, exist_ok=True)
assert (EXPORT_DIR / "metadata.json").exists(), f"Falta la exportación {EXPORT_DIR}: ejecutar avance2 con TARGET_FS = {TARGET_FS}"

def horizons_for(seconds, fs=TARGET_FS, max_heads=MAX_PRED_HEADS):
    """Horizontes k (en pasos) hasta `seconds`. Consecutivos si caben en `max_heads`; si no, espaciados."""
    n = max(1, int(round(seconds * fs)))
    if n <= max_heads:
        return tuple(range(1, n + 1))
    return tuple(int(k) for k in np.unique(np.round(np.linspace(n / max_heads, n, max_heads)).astype(int)))

HORIZONS = {name: horizons_for(s) for name, s in HORIZONS_S.items()}
print("PyTorch:", torch.__version__, "| dispositivo:", DEVICE, "| QUICK:", QUICK)
print("Exportación:", EXPORT_DIR.relative_to(REPO_DIR))
print("Corridas:", RUN_DIR.relative_to(REPO_DIR))
display(pd.DataFrame([
    {"config": name, "horizontes k": h, "nº W_k": len(h),
     "desde (ms)": round(1000 * h[0] / TARGET_FS), "hasta (ms)": round(1000 * h[-1] / TARGET_FS)}
    for name, h in HORIZONS.items()
]).set_index("config"))

# 2. Datos exportados por el Avance 2

Se leen la serie continua (`%MVC` sin escalar), las etiquetas por muestra y los índices de ventanas, y se aplica el escalador guardado en `metadata.json`. **No se recalcula nada**: el escalador se ajustó sólo con ventanas de entrenamiento.

- **CPC:** ventanas sin etiqueta. Con `recortes_aleatorios`, cada batch toma posiciones de inicio al azar entre todas las válidas de los ensayos de entrenamiento; con `indice_fijo`, usa sólo las ventanas de `cpc_index` (salto de 2 s).
- **Clasificación:** `supervised_index` (salto de 0.5 s), etiqueta del **instante final**. La sonda lineal se ajusta con una muestra estratificada de entrenamiento (`PROBE_TRAIN_MAX`) y se evalúa con **todas** las ventanas de validación (sujetos 11–12).

La serie completa se sube una sola vez al dispositivo y las ventanas se recortan ahí mismo.

In [ ]:
meta = json.loads((EXPORT_DIR / "metadata.json").read_text())
assert meta["target_fs"] == TARGET_FS and meta["label_rule"] == "final_instant"
with np.load(EXPORT_DIR / "series.npz", allow_pickle=False) as stored:
    signals_raw, labels_series, offsets = stored["signals"], stored["labels"], stored["offsets"]

prep = meta["preprocessing"]
lower, upper, center, scale = (np.asarray(prep[k], dtype=np.float32) for k in ("lower", "upper", "center", "scale"))
signals_scaled = ((np.log1p(np.clip(signals_raw, lower, upper)) - center) / scale).astype(np.float32)
if CHANNEL_MODE == "pares4":
    signals_scaled = signals_scaled.reshape(-1, 4, 2).mean(axis=2)
N_CH = signals_scaled.shape[1]
WIN, TAIL = meta["window_len"], meta["tail_steps"]
LABELS = meta["label_order"]

trials = pd.read_csv(EXPORT_DIR / "trials.csv")
cpc_index = pd.read_csv(EXPORT_DIR / "cpc_index.csv.gz")
sup_index = pd.read_csv(EXPORT_DIR / "supervised_index.csv.gz")
assert (trials["offset"].to_numpy() == offsets[:-1]).all()

S_T = torch.from_numpy(np.ascontiguousarray(signals_scaled.T)).to(DEVICE)   # (canales, muestras)
WIN_RANGE = torch.arange(WIN, device=DEVICE)

def starts_of(frame):
    return offsets[frame["trial_idx"].to_numpy()] + frame["start_step"].to_numpy()

def get_windows(starts):
    """Posiciones absolutas de inicio -> tensor (B, canales, WIN) en el dispositivo."""
    idx = torch.as_tensor(np.asarray(starts), device=DEVICE)[:, None] + WIN_RANGE
    return S_T[:, idx].permute(1, 0, 2)

def eval_set(frame, cap=None):
    if cap is not None and len(frame) > cap:
        frame = frame.groupby("target", group_keys=False).sample(frac=cap / len(frame), random_state=SEED_DATA)
    return {"frame": frame.reset_index(drop=True), "starts": starts_of(frame), "y": frame["target"].to_numpy()}

sup_train = sup_index[sup_index["split"] == "train"]
PROBE_TRAIN = eval_set(sup_train, PROBE_TRAIN_MAX)
VAL = eval_set(sup_index[sup_index["split"] == "validation"], VAL_CAP)
TEST = eval_set(sup_index[sup_index["split"] == "test"])          # no se usa hasta la sección 12
VAL_CPC_STARTS = starts_of(cpc_index[cpc_index["split"] == "validation"])
N_STEPS = len(cpc_index[cpc_index["split"] == "train"]) // BATCH_SIZE
if STEPS_CAP:
    N_STEPS = min(N_STEPS, STEPS_CAP)

counts = pd.crosstab(sup_index["split"], sup_index["target"]).rename(columns=dict(enumerate(LABELS)))
display(counts.reindex(["train", "validation", "test"]))
print(f"{TARGET_FS} Hz | ventana {WIN} pasos ({WIN / TARGET_FS:g} s) | cola {TAIL} pasos | canales: {N_CH} ({CHANNEL_MODE})")
print(f"CPC: {N_STEPS} pasos por época de {BATCH_SIZE} | sonda: {len(PROBE_TRAIN['y']):,} ventanas de train, "
      f"{len(VAL['y']):,} de validación")

In [ ]:
class CPCBatchSampler:
    """Posiciones de inicio para los batches de CPC.

    sampling = "recortes_aleatorios": cualquier inicio válido de los ensayos del split.
    sampling = "indice_fijo": sólo los inicios de cpc_index (salto CPC_HOP).
    batch_mode = "mezclado": el batch mezcla sujetos (los negativos son de otras personas).
    batch_mode = "mismo_sujeto": todo el batch es de un sujeto, elegido con probabilidad proporcional a sus datos.
    """

    def __init__(self, split, batch_mode, sampling, seed, batch_size=BATCH_SIZE):
        self.rng = np.random.default_rng(seed)
        self.batch_mode, self.batch_size = batch_mode, batch_size
        if sampling == "recortes_aleatorios":
            rows = trials[trials["split"] == split]
            starts = [np.arange(r.offset, r.offset + r.n_steps - WIN + 1) for r in rows.itertuples() if r.n_steps >= WIN]
            subjects = [np.full(r.n_steps - WIN + 1, r.subject) for r in rows.itertuples() if r.n_steps >= WIN]
            self.starts, self.subjects = np.concatenate(starts), np.concatenate(subjects)
        elif sampling == "indice_fijo":
            frame = cpc_index[cpc_index["split"] == split]
            self.starts, self.subjects = starts_of(frame), frame["subject"].to_numpy()
        else:
            raise ValueError(sampling)
        self.by_subject = {s: self.starts[self.subjects == s] for s in np.unique(self.subjects)}
        sizes = np.array([len(v) for v in self.by_subject.values()], dtype=float)
        self.subject_p = sizes / sizes.sum()

    def batch(self):
        if self.batch_mode == "mezclado":
            return self.rng.choice(self.starts, self.batch_size, replace=False)
        subject = self.rng.choice(list(self.by_subject), p=self.subject_p)
        pool = self.by_subject[subject]
        return self.rng.choice(pool, self.batch_size, replace=len(pool) < self.batch_size)

for mode in BATCH_MODES:
    sampler = CPCBatchSampler("train", mode, CPC_SAMPLING, seed=0)
    b = sampler.batch()
    print(f"{mode:13s}: {len(sampler.starts):,} inicios posibles | sujetos en un batch: "
          f"{len(np.unique(sampler.subjects[np.searchsorted(sampler.starts, b)]))}")

# 3. Modelo

Piezas de CPC (las mismas de `01_Pipeline_CPC.ipynb`, con la configuración del artículo de Haresamudram et al., 2020):

| Pieza | Qué hace | Forma |
|---|---|---|
| `CausalEncoder` (`g_enc`) | 3 convoluciones 1D (32, 64, 128 canales, kernel 3, sin *stride*) | `x (B, 8, T) → z (B, 128, T)` |
| `g_ar` | Resume el pasado de forma causal: GRU de 2 capas, o 2 bloques sLSTM / mLSTM | `z (B, T, 128) → c (B, T, 256)` |
| `Predictor` | Una matriz `W_k` por horizonte: predice `z_{t+k}` desde `c_t` | `c_t (B, 256) → (n_horizontes, B, 128)` |
| `InfoNCE` | Para cada `k`, el positivo es el `z_{t+k}` de la misma ventana y los negativos son los de las otras ventanas del batch | pérdida escalar |

`CPCConfig` reúne los hiperparámetros; `pred_horizons` permite elegir qué horizontes se predicen.

In [ ]:
@dataclass
class CPCConfig:
    """
    Hiperparámetros de CPC (los mismos del notebook 01), más `pred_horizons`:
    la lista de horizontes k que se predicen. Vacía = k = 1..n_pred_steps, el comportamiento original.
    """
    # --- entrada ---
    n_channels: int = 6

    # --- g_enc: encoder convolucional ---
    enc_channels: tuple = (32, 64, 128)
    enc_kernel_size: int = 3
    enc_dropout: float = 0.2

    # --- g_ar: modelo autorregresivo ---
    ar_hidden_size: int = 256
    ar_num_layers: int = 2
    ar_dropout: float = 0.2

    # --- predicción / pérdida contrastiva ---
    n_pred_steps: int = 12                  # K: número de horizontes
    n_anchors: int = 4                      # nº de instantes de anclaje por batch de entrenamiento
    pred_horizons: tuple = ()               # NUEVO: horizontes k (en pasos); vacío = 1..n_pred_steps

    @property
    def horizons(self) -> tuple:
        return tuple(self.pred_horizons) or tuple(range(1, self.n_pred_steps + 1))

    @property
    def z_dim(self) -> int:
        return self.enc_channels[-1]

    @property
    def c_dim(self) -> int:
        return self.ar_hidden_size

### 3.1 Encoder causal

El encoder de CPC usa convoluciones de kernel 3 que conservan la longitud de la señal. Si el relleno es simétrico, cada `z_t` depende de `x_{t-1}, x_t, x_{t+1}` en la primera capa, y tras tres capas, de hasta `x_{t+3}`. A 12 Hz eso son 250 ms de futuro, justo dentro del rango que CPC debe predecir: el modelo podría "copiar" en lugar de predecir. `CausalEncoder` mantiene la misma arquitectura y número de parámetros, pero rellena sólo a la izquierda, de modo que `z_t` depende de `x_{t-3..t}`. La sección 4 lo verifica desde la entrada.

In [ ]:
class CausalEncoder(nn.Module):
    """g_enc causal: bloques Conv1d -> ReLU -> Dropout sin stride (Haresamudram et al., 2020),
    con relleno sólo a la izquierda: z_t depende de x_{t-3..t}, nunca del futuro.

        x: (B, n_channels, T)  -->  z: (B, z_dim, T)
    """

    def __init__(self, cfg: CPCConfig):
        super().__init__()
        k = cfg.enc_kernel_size
        self.left_pad = k - 1
        in_channels = (cfg.n_channels,) + cfg.enc_channels[:-1]
        self.convs = nn.ModuleList([nn.Conv1d(c_in, c_out, kernel_size=k) for c_in, c_out in zip(in_channels, cfg.enc_channels)])
        self.dropout = nn.Dropout(cfg.enc_dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        z = x
        for conv in self.convs:
            z = self.dropout(F.relu(conv(F.pad(z, (self.left_pad, 0)))))
        return z

### 3.2 GRU, predictor, InfoNCE y modelo completo

`AutoRegressive` es la GRU del artículo de CPC. `Predictor` e `InfoNCE` aceptan una lista de horizontes (`cfg.horizons`), y `CPCModel` une todas las piezas: `encode(x)` devuelve `z` y `c`, e `info_nce(x)` calcula la pérdida en `n_anchors` instantes de anclaje elegidos al azar.

In [ ]:
class AutoRegressive(nn.Module):
    """
    g_ar: modelo autorregresivo genérico (GRU multicapa).

        z: (B, L, z_dim)  -->  c: (B, L, c_dim)

    Los parámetros de GRU se especifican en la clase cfg. De acuerdo al paper: 
    - input_size: 128
    - hidden_Size: 256
    - num_layer: 2

    c[:, t] resume z[:, :t+1] de forma causal (una GRU nunca mira el futuro),
    que es la propiedad necesaria para usar c_t como predictor de z_{t+k}.

    Arquitectura (Haresamudram, Essa & Plötz, 2020): GRU de 2 capas con 256
    unidades ocultas y dropout entre capas.
    """

    def __init__(self, cfg: CPCConfig):
        super().__init__()
        self.gru = nn.GRU(
            input_size=cfg.z_dim,
            hidden_size=cfg.ar_hidden_size,
            num_layers=cfg.ar_num_layers,
            dropout=cfg.ar_dropout if cfg.ar_num_layers > 1 else 0.0,  # PyTorch exige 0 si num_layers==1
            batch_first=True,
        )

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        c, _ = self.gru(z)        # c: (B, L, c_dim) - Hace un loop sobre todo el eje temporal toma el c_{t-1} e incorpora el nuevo paso z_t
        return c

In [ ]:
class Predictor(nn.Module):
    """
    Cabezas de predicción log-bilineal: una W_k (sin sesgo) por cada horizonte de cfg.horizons.

        c_t: (B, c_dim) -> (n_horizontes, B, z_dim),  con pred_k = W_k c_t ≈ z_{t+k}
    """

    def __init__(self, cfg: CPCConfig):
        super().__init__()
        self.horizons = cfg.horizons
        self.heads = nn.ModuleList([nn.Linear(cfg.c_dim, cfg.z_dim, bias=False) for _ in self.horizons])

    def forward(self, c_t: torch.Tensor) -> torch.Tensor:
        return torch.stack([head(c_t) for head in self.heads], dim=0)


class InfoNCE(nn.Module):
    """
    Pérdida contrastiva de CPC (Ec. 4) con negativos de minibatch, para cada horizonte de la lista.
    Para el horizonte k, el positivo de la secuencia i es z_{t+k}^(i) y los negativos son los
    z_{t+k}^(j) (j != i) de las otras secuencias del batch.
    """

    def __init__(self, horizons: tuple):
        super().__init__()
        self.horizons = tuple(horizons)

    def forward(self, preds: torch.Tensor, z: torch.Tensor, t: int):
        """
        preds: (n_horizontes, B, z_dim) · z: (B, L, z_dim) · t: instante de anclaje
        Devuelve (loss_mean, loss_per_k, acc_per_k), los dos últimos de forma (n_horizontes,).
        """
        H, B, _ = preds.shape
        target = torch.arange(B, device=preds.device)
        loss_per_k = torch.zeros(H, device=preds.device)
        acc_per_k = torch.zeros(H, device=preds.device)
        for j, k in enumerate(self.horizons):
            logits = preds[j] @ z[:, t + k, :].t()          # (B, B): S_ij = pred_i · z_{t+k}^(j)
            loss_per_k[j] = F.cross_entropy(logits, target)
            acc_per_k[j] = (logits.argmax(dim=1) == target).float().mean()
        return loss_per_k.mean(), loss_per_k, acc_per_k.detach()


class CPCModel(nn.Module):
    """Modelo CPC completo (Encoder + g_ar + Predictor + InfoNCE), con horizontes de cfg.horizons."""

    def __init__(self, cfg: CPCConfig):
        super().__init__()
        self.cfg = cfg
        self.encoder = CausalEncoder(cfg)          # encoder causal (sección 3.1)
        self.ar = AutoRegressive(cfg)
        self.predictor = Predictor(cfg)
        self.info_nce_loss = InfoNCE(cfg.horizons)

    def encode(self, x: torch.Tensor):
        """x: (B, n_channels, T) -> z: (B, T, z_dim), c: (B, T, c_dim)"""
        z = self.encoder(x).transpose(1, 2)
        return z, self.ar(z)

    def anchor_range(self, L: int):
        lo, hi = max(1, L // 8), L - max(self.cfg.horizons)   # deja sitio para leer z_{t+k_máx}
        if hi <= lo:
            raise ValueError(f"Ventana T={L} demasiado corta para el horizonte {max(self.cfg.horizons)}")
        return lo, hi

    def info_nce(self, x: torch.Tensor, anchors=None):
        z, c = self.encode(x)
        if anchors is None:
            anchors = torch.randint(*self.anchor_range(z.shape[1]), size=(self.cfg.n_anchors,)).tolist()
        losses, accs = [], []
        for t in anchors:
            _, loss_per_k, acc_per_k = self.info_nce_loss(self.predictor(c[:, t, :]), z, t)
            losses.append(loss_per_k)
            accs.append(acc_per_k)
        loss_per_k = torch.stack(losses).mean(dim=0)
        acc_per_k = torch.stack(accs).mean(dim=0)
        return loss_per_k.mean(), loss_per_k, acc_per_k

### 3.3 xLSTM: capa lineal block-diagonal

Implementación propia en PyTorch de xLSTM, siguiendo las ecuaciones de Beck et al. (2024). Se escribe desde cero (en lugar de usar la librería oficial `xlstm`) porque sus kernels están pensados para CUDA y porque así cada ecuación del artículo queda visible en el código.

xLSTM divide las celdas en **heads**: la recurrencia de sLSTM y las proyecciones `q, k, v` de mLSTM son matrices *block-diagonal*, de modo que cada bloque sólo mezcla sus propias dimensiones.

In [ ]:
class BlockDiagonalLinear(nn.Module):
    """
    Capa lineal con matriz de pesos block-diagonal: cada "head" (bloque) solo mezcla sus propias
    dimensiones. Es la forma en que xLSTM limita la mezcla entre celdas (sLSTM) o abarata las
    proyecciones q, k, v (mLSTM).

        x: (..., dim) -> (..., dim)     con n_blocks bloques de tamaño dim // n_blocks
    """

    def __init__(self, dim: int, n_blocks: int, bias: bool = True):
        super().__init__()
        assert dim % n_blocks == 0, "dim debe ser múltiplo de n_blocks"
        self.n_blocks, self.block = n_blocks, dim // n_blocks
        self.weight = nn.Parameter(torch.randn(n_blocks, self.block, self.block) / math.sqrt(self.block))
        self.bias = nn.Parameter(torch.zeros(dim)) if bias else None

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        shape = x.shape
        x = x.reshape(*shape[:-1], self.n_blocks, self.block)
        y = torch.einsum("...nd,ned->...ne", x, self.weight).reshape(shape)
        return y if self.bias is None else y + self.bias

### 3.4 Celda sLSTM (§2.2 del artículo)

La LSTM clásica usa compuertas sigmoides (entre 0 y 1). sLSTM cambia la compuerta de entrada a **exponencial**, `i_t = exp(ĩ_t)`, para que una entrada nueva pueda "pesar más" que todo lo acumulado y así **revisar decisiones de memoria**. Como `exp` crece sin límite, se añaden:

- un **normalizador** `n_t = f_t·n_{t-1} + i_t`, y la salida es `h_t = o_t · c_t / n_t` (Ec. 35–36);
- un **estabilizador** `m_t = max(log f_t + m_{t-1}, log i_t)` que se resta dentro de las exponenciales (Ec. 15–17). No cambia el resultado: `c_t` y `n_t` se escalan igual y su cociente es el mismo (Apéndice A.2).

La recurrencia `R·h_{t-1}` es block-diagonal por head (**memory mixing** dentro de cada head). Por eso sLSTM no se puede paralelizar en el tiempo: recorre `t = 1..T` en un bucle, como la GRU, y es la variante más lenta.

In [ ]:
class sLSTMCell(nn.Module):
    """
    Celda sLSTM (Beck et al. 2024, Ec. 34-40, con estabilización Ec. 15-17).

        x: (B, T, dim) -> h: (B, T, dim)

    Diferencias con la LSTM clásica:
      - compuerta de entrada EXPONENCIAL  i_t = exp(ĩ_t)  (permite "revisar" decisiones de memoria)
      - estado normalizador  n_t = f_t n_{t-1} + i_t,  y  h_t = o_t * c_t / n_t
      - estado estabilizador m_t para que exp() no desborde (no cambia el resultado)
      - memory mixing por heads: la recurrencia R h_{t-1} es block-diagonal (n_heads bloques)

    Es recurrente (no paralelizable): recorre el tiempo paso a paso, igual que la GRU.
    """

    def __init__(self, dim: int, n_heads: int):
        super().__init__()
        self.dim = dim
        self.W = nn.Linear(dim, 4 * dim)                                     # entrada -> z, i, f, o
        self.R = nn.ModuleList([BlockDiagonalLinear(dim, n_heads, bias=False) for _ in range(4)])
        with torch.no_grad():                                                # sesgo de la forget gate en [3, 6]
            self.W.bias[2 * dim:3 * dim] = torch.linspace(3.0, 6.0, dim)    # (Tabla 2 del paper): al inicio "recuerda"

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, D = x.shape
        Wx = self.W(x)                                                       # parte de la entrada, para todo t a la vez
        h = c = n = m = x.new_zeros(B, D)
        outputs = []
        for t in range(T):
            Rh = torch.cat([r(h) for r in self.R], dim=-1)                   # parte recurrente (memory mixing)
            z_pre, i_pre, f_pre, o_pre = (Wx[:, t] + Rh).chunk(4, dim=-1)
            log_f = F.logsigmoid(f_pre)                                      # forget gate sigmoide, en log
            m_new = torch.maximum(log_f + m, i_pre)                          # Ec. 15: estabilizador
            i_gate = torch.exp(i_pre - m_new)                                # Ec. 16: i' (exponencial estabilizada)
            f_gate = torch.exp(log_f + m - m_new)                            # Ec. 17: f'
            c = f_gate * c + i_gate * torch.tanh(z_pre)                      # Ec. 34: estado de celda
            n = f_gate * n + i_gate                                          # Ec. 35: normalizador
            h = torch.sigmoid(o_pre) * c / n                                 # Ec. 36: estado oculto
            m = m_new
            outputs.append(h)
        return torch.stack(outputs, dim=1)

### 3.5 Bloque sLSTM (Fig. 10 del artículo)

La celda se envuelve en un bloque residual con *post up-projection* (como un Transformer): primero resume el pasado en la dimensión original y después aplica un MLP con compuerta que proyecta a 4/3 de la dimensión y regresa. Se omite la convolución opcional de la Fig. 10.

In [ ]:
class sLSTMBlock(nn.Module):
    """
    Bloque residual sLSTM con "post up-projection" (Fig. 10 del paper, sin la convolución opcional):

        x -> LayerNorm -> sLSTM (n_heads) -> GroupNorm -> (+x)
          -> LayerNorm -> MLP con compuerta (factor 4/3, GeLU) -> (+x)
    """

    def __init__(self, dim: int, n_heads: int, dropout: float, proj_factor: float = 4 / 3):
        super().__init__()
        inner = int(round(proj_factor * dim))
        self.ln1, self.ln2 = nn.LayerNorm(dim), nn.LayerNorm(dim)
        self.cell = sLSTMCell(dim, n_heads)
        self.gn = nn.GroupNorm(n_heads, dim)                                 # LayerNorm por head
        self.up_a, self.up_b = nn.Linear(dim, inner), nn.Linear(dim, inner)
        self.down = nn.Linear(inner, dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.cell(self.ln1(x))
        h = self.gn(h.reshape(-1, h.shape[-1])).reshape(h.shape)
        x = x + self.drop(h)
        y = self.ln2(x)
        return x + self.drop(self.down(self.up_a(y) * F.gelu(self.up_b(y))))

### 3.6 mLSTM (§2.3 y Apéndice A.3 del artículo)

mLSTM cambia la memoria escalar `c_t` por una **matriz** `C_t` (de `dh × dh` por head) que guarda pares clave–valor con la **regla de covarianza** `C_t = f_t·C_{t-1} + i_t·v_t·k_tᵀ`; para leer, se multiplica por una consulta: `h̃_t = C_t·q_t / max(|n_tᵀ q_t|, 1)` (Ec. 19–21). Así aumenta la capacidad de memoria.

Como mLSTM **no tiene recurrencia oculta** (`h_{t-1}` no entra en las compuertas), la misma cuenta se puede escribir en **forma paralela**: una matriz `T × T` de scores `q·kᵀ/√d` multiplicada por una matriz de compuertas `D` triangular inferior (causal). Se implementan las dos formas: la paralela se usa para entrenar y la recurrente sólo para comprobar que dan lo mismo (sección 4).

In [ ]:
def mlstm_parallel(q, k, v, i_pre, f_pre, n_heads):
    """
    mLSTM en su forma PARALELA estabilizada (Apéndice A.3, Ec. 79-86): procesa todos los
    instantes a la vez con una matriz T x T, como la atención, pero con compuertas.

        q, k, v      : (B, T, D)        D = n_heads * dh
        i_pre, f_pre : (B, T, n_heads)  pre-activaciones de las compuertas de entrada y olvido
        -> h̃         : (B, T, D)

    Es equivalente a la recurrencia  C_t = f_t C_{t-1} + i_t v_t k_tᵀ,  n_t = f_t n_{t-1} + i_t k_t,
    h̃_t = C_t q_t / max(|n_tᵀ q_t|, 1)  (Ec. 19-21), y causal: el instante i solo usa j <= i.
    """
    B, T, D = q.shape
    dh = D // n_heads
    q, k, v = [x.view(B, T, n_heads, dh).transpose(1, 2) for x in (q, k, v)]   # (B, H, T, dh)
    i_pre, f_pre = i_pre.transpose(1, 2), f_pre.transpose(1, 2)                # (B, H, T)

    log_f = F.logsigmoid(f_pre)
    cum = torch.cumsum(log_f, dim=-1)
    log_F = cum.unsqueeze(-1) - cum.unsqueeze(-2)              # [i, j] = sum_{k=j+1..i} log f_k   (Ec. 101)
    causal = torch.ones(T, T, dtype=torch.bool, device=q.device).tril()
    log_D = (log_F + i_pre.unsqueeze(-2)).masked_fill(~causal, float("-inf"))  # Ec. 79: log F + Ĩ
    m = log_D.amax(dim=-1, keepdim=True)                       # Ec. 80: estabilizador por fila
    D_mat = torch.exp(log_D - m)                               # Ec. 81
    C = (q @ k.transpose(-2, -1)) / math.sqrt(dh) * D_mat      # Ec. 82: scores qkᵀ/√d con compuertas
    n = torch.maximum(C.sum(dim=-1, keepdim=True).abs(), torch.exp(-m))   # Ec. 83-84: normalizador
    h = (C / n) @ v                                            # Ec. 85-86
    return h.transpose(1, 2).reshape(B, T, D)


def mlstm_recurrent(q, k, v, i_pre, f_pre, n_heads):
    """La misma mLSTM en forma RECURRENTE (Ec. 19-21, estabilizada). Solo se usa para verificar."""
    B, T, D = q.shape
    dh = D // n_heads
    q, k, v = [x.view(B, T, n_heads, dh) for x in (q, k, v)]
    k = k / math.sqrt(dh)                                      # Ec. 23: k escalado por 1/√d
    C = q.new_zeros(B, n_heads, dh, dh)
    n = q.new_zeros(B, n_heads, dh)
    m = q.new_full((B, n_heads), float("-inf"))
    out = []
    for t in range(T):
        log_f = F.logsigmoid(f_pre[:, t])
        m_new = torch.maximum(log_f + m, i_pre[:, t])
        i_g = torch.exp(i_pre[:, t] - m_new)[..., None]
        f_g = torch.exp(log_f + m - m_new)[..., None]
        C = f_g[..., None] * C + i_g[..., None] * (v[:, t, :, :, None] @ k[:, t, :, None, :])  # C += i v kᵀ
        n = f_g * n + i_g * k[:, t]
        num = (C @ q[:, t, :, :, None]).squeeze(-1)
        den = torch.maximum((n * q[:, t]).sum(-1, keepdim=True).abs(), torch.exp(-m_new)[..., None])
        out.append(num / den)
        m = m_new
    return torch.stack(out, dim=1).reshape(B, T, D)

### 3.7 Bloque mLSTM (Fig. 11 del artículo)

Bloque con *pre up-projection* (como los State Space Models): primero proyecta a 2× la dimensión y ahí aplica la memoria matricial. Incluye una **convolución causal** de tamaño 4 antes de `q` y `k` (el artículo muestra que es importante, Tabla 8), un *skip* aprendible y una compuerta de salida externa (rama con Swish).

In [ ]:
class mLSTMBlock(nn.Module):
    """
    Bloque residual mLSTM con "pre up-projection" (Fig. 11 del paper):

        x -> LayerNorm -> proyección x2 en dos ramas:
             rama celda : conv causal (k=4) + Swish -> q, k ; (sin conv) -> v ; compuertas i, f
                          -> mLSTM -> GroupNorm -> + skip aprendible
             rama salida: Swish  (compuerta de salida externa, por componente)
          -> producto de ambas ramas -> proyección de vuelta a dim -> (+x)
    """

    def __init__(self, dim: int, n_heads: int, dropout: float, proj_factor: int = 2,
                 conv_kernel: int = 4, qkv_block: int = 4):
        super().__init__()
        inner = proj_factor * dim
        self.n_heads, self.conv_kernel = n_heads, conv_kernel
        self.ln = nn.LayerNorm(dim)
        self.up_cell, self.up_gate = nn.Linear(dim, inner), nn.Linear(dim, inner)
        self.conv = nn.Conv1d(inner, inner, conv_kernel, groups=inner)       # por dimensión (depthwise)
        self.q = BlockDiagonalLinear(inner, inner // qkv_block)
        self.k = BlockDiagonalLinear(inner, inner // qkv_block)
        self.v = BlockDiagonalLinear(inner, inner // qkv_block)
        self.igate, self.fgate = nn.Linear(3 * inner, n_heads), nn.Linear(3 * inner, n_heads)
        with torch.no_grad():
            self.fgate.bias.copy_(torch.linspace(3.0, 6.0, n_heads))         # forget gate: al inicio recuerda
            nn.init.normal_(self.igate.bias, 0.0, 0.1)                       # input gate: N(0, 0.1) (Tabla 2)
        self.gn = nn.GroupNorm(n_heads, inner)
        self.skip = nn.Parameter(torch.ones(inner))                          # "learnable skip" de la Fig. 11
        self.down = nn.Linear(inner, dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = self.ln(x)
        x_cell, x_gate = self.up_cell(y), self.up_gate(y)
        # Convolución CAUSAL: se rellena solo a la izquierda, así el instante t no ve t+1, t+2, ...
        x_conv = F.silu(self.conv(F.pad(x_cell.transpose(1, 2), (self.conv_kernel - 1, 0))).transpose(1, 2))
        q, k, v = self.q(x_conv), self.k(x_conv), self.v(x_cell)
        qkv = torch.cat([q, k, v], dim=-1)
        h = mlstm_parallel(q, k, v, self.igate(qkv), self.fgate(qkv), self.n_heads)
        h = self.gn(h.reshape(-1, h.shape[-1])).reshape(h.shape) + self.skip * x_conv
        return x + self.drop(self.down(h * F.silu(x_gate)))

### 3.8 `g_ar` xLSTM con la interfaz de la GRU

Proyección `z_dim → c_dim` y 2 bloques del mismo tipo (como las 2 capas de la GRU). La salida `c_t` tiene 256 dimensiones, así que `Predictor`, `InfoNCE` y la evaluación no cambian.

`build_model` construye el modelo completo con el `g_ar` elegido; la semilla fija los pesos iniciales, de modo que una misma semilla da la misma red sin entrenar en todas las configuraciones. `summarize` convierte la secuencia `c` de una ventana en un vector según la representación elegida.

In [ ]:
class xLSTMAutoRegressive(nn.Module):
    """
    g_ar alternativo a la GRU, con la MISMA interfaz que AutoRegressive del notebook 01:

        z: (B, T, z_dim) -> c: (B, T, c_dim)

    Proyecta z_t a c_dim y apila `n_blocks` bloques xLSTM residuales (todos sLSTM o todos mLSTM),
    como la GRU apila 2 capas. Todo es causal: c_t solo depende de z_1..z_t.
    """

    def __init__(self, cfg: CPCConfig, kind: str, n_blocks: int = 2, n_heads: int = 4):
        super().__init__()
        assert kind in ("sLSTM", "mLSTM")
        Block = sLSTMBlock if kind == "sLSTM" else mLSTMBlock
        self.proj = nn.Linear(cfg.z_dim, cfg.c_dim)
        self.blocks = nn.ModuleList([Block(cfg.c_dim, n_heads, cfg.ar_dropout) for _ in range(n_blocks)])
        self.ln_out = nn.LayerNorm(cfg.c_dim)

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        x = self.proj(z)
        for block in self.blocks:
            x = block(x)
        return self.ln_out(x)

In [ ]:
def build_model(kind: str, horizons: tuple, seed: int, device=None) -> CPCModel:
    """CPC completo con el g_ar elegido. La semilla fija los pesos iniciales (misma semilla = misma red)."""
    cfg = CPCConfig(n_channels=N_CH, n_pred_steps=len(horizons), n_anchors=4, pred_horizons=tuple(horizons))
    torch.manual_seed(seed)
    model = CPCModel(cfg)
    if kind != "GRU":
        model.ar = xLSTMAutoRegressive(cfg, kind)
    return model.to(device or DEVICE)


def summarize(c: torch.Tensor, representation: str) -> torch.Tensor:
    """c: (B, T, c_dim) -> vector por ventana."""
    if representation == "c_T":
        return c[:, -1]
    if representation == "cola":
        return c[:, -TAIL:].mean(dim=1)
    if representation == "media":
        return c.mean(dim=1)
    if representation == "c_T+media":
        return torch.cat([c[:, -1], c.mean(dim=1)], dim=1)
    raise ValueError(representation)

# 4. Verificaciones y estimación de tiempo

1. **Causalidad de extremo a extremo** para los tres `g_ar`: se perturba la **entrada** sEMG desde `T/2` y nada antes de `T/2` en `c` debe cambiar.
2. **mLSTM paralela = recurrente** con la longitud de ventana de esta frecuencia.
3. **Anclajes válidos:** con `T = WIN` debe quedar sitio para el horizonte más largo de cada configuración.
4. **Dispositivo:** en modo evaluación, el modelo da lo mismo en el dispositivo y en CPU.
5. **Tiempo:** se miden unos pasos de entrenamiento por modelo y se extrapola la etapa 1.

In [ ]:
def sync():
    if DEVICE.type == "mps":
        torch.mps.synchronize()
    elif DEVICE.type == "cuda":
        torch.cuda.synchronize()

rows = []
h_long = HORIZONS[max(HORIZONS, key=lambda n: max(HORIZONS[n]))]
for kind in MODELS:
    model = build_model(kind, h_long, seed=0).eval()
    x = torch.randn(4, N_CH, WIN, device=DEVICE)
    x_mod = x.clone()
    x_mod[:, :, WIN // 2:] += torch.randn_like(x_mod[:, :, WIN // 2:])
    with torch.no_grad():
        _, c, = model.encode(x)
        _, c_mod = model.encode(x_mod)
        past = (c[:, :WIN // 2] - c_mod[:, :WIN // 2]).abs().max().item()
        future = (c[:, WIN // 2:] - c_mod[:, WIN // 2:]).abs().max().item()
        cpu_model = build_model(kind, h_long, seed=0, device="cpu").eval()
        device_gap = (model.encode(x)[1].cpu() - cpu_model.encode(x.cpu())[1]).abs().max().item()
    assert past == 0.0 and future > 0, f"{kind}: fuga temporal ({past:.2e})"
    model.train()
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    sampler = CPCBatchSampler("train", "mezclado", CPC_SAMPLING, seed=0)
    for it in range(5):
        if it == 2:
            sync()
            t0 = time.time()
        loss, _, _ = model.info_nce(get_windows(sampler.batch()))
        opt.zero_grad()
        loss.backward()
        opt.step()
    sync()
    sec_step = (time.time() - t0) / 3
    rows.append({"modelo": kind, "fuga antes de T/2": past, "dif. dispositivo vs CPU": device_gap,
                 "s/paso": sec_step, "min por corrida (máx.)": sec_step * N_STEPS * EPOCHS_MAX / 60})
    del model, cpu_model, opt

q, k_, v = (torch.randn(2, WIN, 16) for _ in range(3))
i_pre, f_pre = torch.randn(2, WIN, 4), torch.randn(2, WIN, 4) + 2
mlstm_gap = (mlstm_parallel(q, k_, v, i_pre, f_pre, 4) - mlstm_recurrent(q, k_, v, i_pre, f_pre, 4)).abs().max().item()
for name, h in HORIZONS.items():
    lo, hi = build_model("GRU", h, 0).anchor_range(WIN)
    print(f"Anclajes {name}: t ∈ [{lo}, {hi})")
print(f"mLSTM paralela vs recurrente: diferencia máxima {mlstm_gap:.1e}")
assert mlstm_gap < 1e-3

check = pd.DataFrame(rows).set_index("modelo")
n_runs_stage1 = len(HORIZONS) * len(BATCH_MODES)
print(f"Etapa 1 (≤ {EPOCHS_MAX} épocas, sin contar evaluaciones): ≈ "
      f"{check['min por corrida (máx.)'].sum() * n_runs_stage1 / 60:.1f} h como máximo")
display(check.round({"s/paso": 3, "min por corrida (máx.)": 1}))

# 5. Herramientas de entrenamiento y evaluación

- **`contrastive_eval`:** InfoNCE en batches fijos de validación (mismas ventanas y anclajes en cada época).
- **`probe_eval`:** sonda lineal (`StandardScaler` + regresión logística balanceada) ajustada con las ventanas de entrenamiento y evaluada en validación, para cada representación pedida. Reporta F1 macro, *balanced accuracy*, F1 por clase y matriz de confusión.
- **`train_cpc`:** entrena CPC con el muestreo y el modo de batch elegidos; cada `EVAL_EVERY` épocas mide el F1 de validación con `EARLY_STOP_REPS` y conserva los pesos de la mejor época. Para tras `PATIENCE` evaluaciones sin mejorar. Al terminar evalúa las cuatro representaciones con los mejores pesos.
- Cada corrida guarda un `.pt` (pesos, configuración, horizontes y estado del preprocesamiento) y un `.json` con el historial y las métricas.

In [ ]:
def metrics(y_true, y_pred):
    ids = list(range(len(LABELS)))
    return {
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "accuracy": accuracy_score(y_true, y_pred),
        "f1_per_class": f1_score(y_true, y_pred, labels=ids, average=None, zero_division=0).tolist(),
        "confusion": confusion_matrix(y_true, y_pred, labels=ids, normalize="true").tolist(),
    }


@torch.no_grad()
def extract_reps(model, starts, reps=REPRESENTATIONS, batch=512):
    model.eval()
    out = {r: [] for r in reps}
    for s in range(0, len(starts), batch):
        _, c = model.encode(get_windows(starts[s:s + batch]))
        for r in reps:
            out[r].append(summarize(c, r).float().cpu().numpy())
    return {r: np.concatenate(v) for r, v in out.items()}


def fit_probe(X, y):
    return make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000, class_weight="balanced")).fit(X, y)


def probe_eval(model, reps, eval_sets, return_preds=False):
    """{rep: {split: métricas}}; la sonda se ajusta siempre con PROBE_TRAIN."""
    train_reps = extract_reps(model, PROBE_TRAIN["starts"], reps)
    eval_reps = {name: extract_reps(model, es["starts"], reps) for name, es in eval_sets.items()}
    out, preds = {}, {}
    for r in reps:
        clf = fit_probe(train_reps[r], PROBE_TRAIN["y"])
        out[r] = {name: metrics(es["y"], clf.predict(eval_reps[name][r])) for name, es in eval_sets.items()}
        preds[r] = {name: clf.predict(eval_reps[name][r]) for name in eval_sets}
    return (out, preds) if return_preds else out


@torch.no_grad()
def contrastive_eval(model, n_batches=20):
    model.eval()
    g = np.random.default_rng(SEED_DATA)
    lo, hi = model.anchor_range(WIN)
    losses, accs = [], []
    for _ in range(min(n_batches, len(VAL_CPC_STARTS) // BATCH_SIZE)):
        starts = g.choice(VAL_CPC_STARTS, BATCH_SIZE, replace=False)
        anchors = g.integers(lo, hi, size=model.cfg.n_anchors).tolist()
        loss, _, acc = model.info_nce(get_windows(starts), anchors=anchors)
        losses.append(loss.item())
        accs.append(acc.cpu())
    return float(np.mean(losses)), torch.stack(accs).mean(0).numpy()


def run_name(kind, hname, batch_mode, seed):
    return f"{kind}_{hname}_{batch_mode}_s{seed}"


def save_run(path_json, model, result):
    torch.save({"state_dict": model.state_dict(), "cfg": asdict(model.cfg), "kind": result["model"],
                "channel_mode": CHANNEL_MODE, "preprocessing": prep}, path_json.with_suffix(".pt"))
    path_json.write_text(json.dumps(result))


def train_cpc(kind, hname, batch_mode, seed):
    horizons = HORIZONS[hname]
    model = build_model(kind, horizons, seed)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    sampler = CPCBatchSampler("train", batch_mode, CPC_SAMPLING, seed)
    torch.manual_seed(seed)
    best = {"score": -1.0, "epoch": 0, "state": copy.deepcopy(model.state_dict())}
    bad, history, t_start = 0, [], time.time()
    for epoch in range(1, EPOCHS_MAX + 1):
        model.train()
        t0, run_loss, run_acc = time.time(), 0.0, 0.0
        for _ in range(N_STEPS):
            loss, _, acc_k = model.info_nce(get_windows(sampler.batch()))
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()
            run_loss += loss.item()
            run_acc += acc_k.mean().item()
        val_loss, val_acc_k = contrastive_eval(model)
        row = {"epoch": epoch, "train_loss": run_loss / N_STEPS, "train_acc": run_acc / N_STEPS,
               "val_loss": val_loss, "val_acc": float(val_acc_k.mean()), "secs": time.time() - t0}
        msg = f"   época {epoch:2d} | train {row['train_loss']:.3f} | val {val_loss:.3f} acc {row['val_acc']:.3f}"
        if epoch % EVAL_EVERY == 0 or epoch == EPOCHS_MAX:
            ev = probe_eval(model, EARLY_STOP_REPS, {"validación": VAL})
            for r in EARLY_STOP_REPS:
                row[f"f1_val_{r}"] = ev[r]["validación"]["f1_macro"]
            score = max(row[f"f1_val_{r}"] for r in EARLY_STOP_REPS)
            msg += " | F1 val " + ", ".join(f"{r} {row[f'f1_val_{r}']:.3f}" for r in EARLY_STOP_REPS)
            if score > best["score"] + 1e-4:
                best, bad = {"score": score, "epoch": epoch, "state": copy.deepcopy(model.state_dict())}, 0
                msg += " *"
            else:
                bad += 1
        history.append(row)
        print(msg)
        if bad >= PATIENCE:
            print(f"   parada temprana en la época {epoch} (mejor: {best['epoch']})")
            break
    model.load_state_dict(best["state"])
    final = probe_eval(model, REPRESENTATIONS, {"validación": VAL})
    return model, {"model": kind, "horizons_name": hname, "horizons": list(horizons), "batch_mode": batch_mode,
                   "seed": seed, "sampling": CPC_SAMPLING, "best_epoch": best["epoch"], "history": history,
                   "val_acc_k": val_acc_k.tolist(), "minutes": (time.time() - t_start) / 60,
                   "params_ar": sum(p.numel() for p in model.ar.parameters()), "eval": final}


def run_cpc(kind, hname, batch_mode, seed):
    path = RUN_DIR / "cpc" / f"{run_name(kind, hname, batch_mode, seed)}.json"
    if path.exists() and not RERUN:
        r = json.loads(path.read_text())
        best_rep = max(REPRESENTATIONS, key=lambda rep: r["eval"][rep]["validación"]["f1_macro"])
        print(f"{path.stem}: corrida guardada | mejor época {r['best_epoch']} de {len(r['history'])} | "
              f"{r['minutes']:.0f} min | F1 val {best_rep} {r['eval'][best_rep]['validación']['f1_macro']:.3f}")
        return
    print(f"{path.stem}: horizontes {HORIZONS[hname]}")
    model, result = train_cpc(kind, hname, batch_mode, seed)
    save_run(path, model, result)
    best_rep = max(REPRESENTATIONS, key=lambda r: result["eval"][r]["validación"]["f1_macro"])
    print(f"   -> {result['minutes']:.0f} min | mejor época {result['best_epoch']} | "
          f"F1 val {best_rep} {result['eval'][best_rep]['validación']['f1_macro']:.3f}")
    del model


def run_random(kind, seed):
    """La misma arquitectura sin entrenar (los horizontes no influyen en el encoder ni en g_ar)."""
    path = RUN_DIR / "sin_entrenar" / f"{kind}_s{seed}.json"
    if path.exists() and not RERUN:
        return
    model = build_model(kind, HORIZONS[next(iter(HORIZONS))], seed)
    path.write_text(json.dumps({"model": kind, "seed": seed, "eval": probe_eval(model, REPRESENTATIONS, {"validación": VAL})}))
    del model


def collect_results():
    rows = []
    for source, folder in (("CPC", "cpc"), ("sin entrenar", "sin_entrenar")):
        for path in sorted((RUN_DIR / folder).glob("*.json")):
            r = json.loads(path.read_text())
            for rep, by_split in r["eval"].items():
                for split, m in by_split.items():
                    rows.append({"fuente": source, "modelo": r["model"], "horizonte": r.get("horizons_name", "—"),
                                 "batch": r.get("batch_mode", "—"), "semilla": r["seed"], "rep": rep, "split": split,
                                 "época": r.get("best_epoch"), "F1": m["f1_macro"], "bal_acc": m["balanced_accuracy"],
                                 **{f"F1 {l}": f for l, f in zip(LABELS, m["f1_per_class"])}})
    res = pd.DataFrame(rows)
    if len(res):
        res.to_csv(SUMMARY_CSV, index=False)
    return res

# 6. Etapa 1 — barrido con una semilla

Todas las combinaciones `modelo × horizonte × modo de batch` con la primera semilla, más cada arquitectura sin entrenar con esa misma semilla (mismos pesos iniciales). La tabla muestra el F1 macro de validación (sujetos 11–12) de cada representación y la **ganancia de CPC** sobre la red sin entrenar.

In [ ]:
t_total = time.time()
for kind in MODELS:
    run_random(kind, SEEDS[0])
    for hname in HORIZONS:
        for batch_mode in BATCH_MODES:
            run_cpc(kind, hname, batch_mode, SEEDS[0])
print(f"Etapa 1 completa ({(time.time() - t_total) / 60:.0f} min en esta ejecución)")

In [ ]:
res = collect_results()
val = res[res["split"] == "validación"]
rand1 = val[(val["fuente"] == "sin entrenar") & (val["semilla"] == SEEDS[0])].set_index(["modelo", "rep"])["F1"]
stage1 = val[(val["fuente"] == "CPC") & (val["semilla"] == SEEDS[0])].copy()
stage1["ganancia"] = stage1["F1"].to_numpy() - rand1.reindex(list(zip(stage1["modelo"], stage1["rep"]))).to_numpy()

table1 = stage1.pivot_table(index=["modelo", "horizonte", "batch"], columns="rep", values=["F1", "ganancia"])
table1 = table1.reindex(columns=pd.MultiIndex.from_product([["F1", "ganancia"], list(REPRESENTATIONS)]))
print("Sin entrenar (misma semilla):")
display(rand1.unstack("rep").reindex(columns=list(REPRESENTATIONS)).round(3))
print("CPC preentrenado:")
display(table1.style.format("{:.3f}").background_gradient(cmap="Blues", subset=["F1"], vmin=0.2, vmax=0.6)
        .map(lambda v: "" if pd.isna(v) else f"color: {'#1a7f37' if v > 0 else '#c0392b'}", subset=["ganancia"]))

fig, ax = plt.subplots(figsize=(13, 4))
best_per_config = stage1.loc[stage1.groupby(["modelo", "horizonte", "batch"])["F1"].idxmax()]
labels_x = [f"{r.modelo}\n{r.horizonte} · {r.batch}\n({r.rep})" for r in best_per_config.itertuples()]
colors = {"GRU": "#2a78d6", "sLSTM": "#eb6834", "mLSTM": "#1baf7a"}
ax.bar(range(len(best_per_config)), best_per_config["F1"], color=[colors[m] for m in best_per_config["modelo"]])
for i, r in enumerate(best_per_config.itertuples()):
    ax.hlines(rand1.loc[r.modelo].max(), i - 0.4, i + 0.4, colors="black", linestyles=":", lw=1.5)
    ax.text(i, r.F1 + 0.005, f"{r.F1:.3f}", ha="center", fontsize=8)
ax.set(xticks=range(len(best_per_config)), xticklabels=labels_x, ylabel="F1 macro (validación)",
       title="Etapa 1: mejor representación por configuración (línea punteada = misma red sin entrenar, mejor representación)")
ax.tick_params(axis="x", labelsize=7)
plt.tight_layout()
plt.show()

### Historial de entrenamiento de la etapa 1

Se lee del `.json` de cada corrida, así que se muestra aunque las corridas se hayan reutilizado. Izquierda: pérdida InfoNCE de validación por época (sólo es comparable entre corridas con el mismo horizonte: predecir más lejos es más difícil). Derecha: F1 de validación de la sonda lineal en las épocas evaluadas (el mayor entre `EARLY_STOP_REPS`); el punto marca la época cuyos pesos se conservaron. Línea continua = `mismo_sujeto`, discontinua = `mezclado`.

In [ ]:
run_rows = []
fig, axes = plt.subplots(1, 2, figsize=(15, 4.2))
for path in sorted((RUN_DIR / "cpc").glob(f"*_s{SEEDS[0]}.json")):
    r = json.loads(path.read_text())
    h = pd.DataFrame(r["history"])
    ls = "-" if r["batch_mode"] == "mismo_sujeto" else "--"
    label = f"{r['model']} {r['horizons_name']} · {r['batch_mode']}"
    axes[0].plot(h["epoch"], h["val_loss"], ls=ls, color=colors[r["model"]], alpha=0.8, label=label)
    f1_cols = [f"f1_val_{rep}" for rep in EARLY_STOP_REPS if f"f1_val_{rep}" in h]
    ev = h.dropna(subset=f1_cols)
    f1_curve = ev[f1_cols].max(axis=1)
    axes[1].plot(ev["epoch"], f1_curve, ls=ls, marker=".", color=colors[r["model"]], alpha=0.8, label=label)
    axes[1].scatter(r["best_epoch"], f1_curve[ev["epoch"] == r["best_epoch"]].max(), color=colors[r["model"]], s=40, zorder=3)
    run_rows.append({"corrida": path.stem, "épocas": len(h), "mejor época": r["best_epoch"], "minutos": r["minutes"],
                     "F1 val (mejor rep.)": max(r["eval"][rep]["validación"]["f1_macro"] for rep in REPRESENTATIONS)})
axes[0].set(title="InfoNCE de validación", xlabel="época", ylabel="pérdida")
axes[1].set(title="F1 macro de validación (sonda lineal)", xlabel="época", ylabel="F1 macro")
axes[1].legend(frameon=False, fontsize=7, ncol=2)
plt.tight_layout()
plt.show()
display(pd.DataFrame(run_rows).set_index("corrida").round({"minutos": 0, "F1 val (mejor rep.)": 3}))

# 7. Etapa 2 — semillas para las mejores configuraciones

Con una sola semilla, diferencias de ±0.02–0.03 en F1 pueden deberse sólo a la inicialización. Las `TOP_K_CONFIGS` mejores configuraciones de la etapa 1 (por su mejor representación en validación) se repiten con todas las `SEEDS`, junto con su red sin entrenar. Se elige la combinación **configuración + representación** con mayor F1 medio de validación.

In [ ]:
ranking = (stage1.groupby(["modelo", "horizonte", "batch"])["F1"].max().sort_values(ascending=False))
top_configs = list(ranking.index[:TOP_K_CONFIGS])
print("Configuraciones elegidas:", top_configs)
for kind, hname, batch_mode in top_configs:
    for seed in SEEDS:
        run_random(kind, seed)
        run_cpc(kind, hname, batch_mode, seed)

res = collect_results()
val = res[res["split"] == "validación"]
cpc_val = val[val["fuente"] == "CPC"].set_index(["modelo", "horizonte", "batch"])
cpc_top = cpc_val.loc[cpc_val.index.isin(top_configs)].reset_index()
summary2 = cpc_top.groupby(["modelo", "horizonte", "batch", "rep"])["F1"].agg(["mean", "std", "count"])
rand_all = val[val["fuente"] == "sin entrenar"].groupby(["modelo", "rep"])["F1"].agg(["mean", "std", "count"])
display(summary2.round(3))
print("Sin entrenar (todas las semillas disponibles):")
display(rand_all.round(3))

BEST = summary2["mean"].idxmax()            # (modelo, horizonte, batch, rep)
BEST_KIND, BEST_H, BEST_BATCH, BEST_REP = BEST
print(f"Mejor: {BEST_KIND}, horizonte {BEST_H}, batch {BEST_BATCH}, representación {BEST_REP} -> "
      f"F1 val {summary2.loc[BEST, 'mean']:.3f} ± {summary2.loc[BEST, 'std']:.3f}")

fig, ax = plt.subplots(figsize=(10, 3.8))
for i, (cfg, g) in enumerate(cpc_top.groupby(["modelo", "horizonte", "batch", "rep"])):
    ax.scatter([i] * len(g), g["F1"], color=colors[cfg[0]], s=25)
    ax.hlines(g["F1"].mean(), i - 0.3, i + 0.3, color="black", lw=1.5)
ticks = [" · ".join(map(str, cfg)) for cfg, _ in cpc_top.groupby(["modelo", "horizonte", "batch", "rep"])]
ax.set(xticks=range(len(ticks)), ylabel="F1 macro (validación)", title="Etapa 2: cada punto es una semilla; la barra, la media")
ax.set_xticklabels(ticks, rotation=60, ha="right", fontsize=7)
plt.tight_layout()
plt.show()

# 8. Diagnósticos

Con el mejor modelo (primera semilla) y su red sin entrenar:

1. **F1 por clase y matriz de confusión** en validación.
2. **F1 según el tiempo desde el cambio de actividad** (`steps_in_segment`). Con etiqueta en el instante final, justo después de un cambio la ventana contiene sobre todo la actividad anterior. Si el modelo acierta mucho antes de que la actividad nueva sea visible, probablemente usa el **orden fijo del protocolo** (N → LF → K → PT → …) y no la señal.
3. **Orden del protocolo:** para cada ventana se busca la **actividad anterior** y se construye un "oráculo de orden" que predice la etiqueta sólo con ella (la transición más frecuente en entrenamiento, sin mirar la señal). Por tramo de tiempo desde el cambio se compara la accuracy del modelo con la del oráculo, cuántas veces el modelo predice la actividad anterior y el recall por clase en el primer medio segundo.
4. **Sonda de sujeto:** regresión logística que predice **quién** es el sujeto (1–10) desde `c_T`, entrenada con la mitad de los ensayos de cada sujeto y evaluada con la otra mitad. Mide cuánta información de la persona conserva la representación. La hipótesis es que, con negativos de otras personas, InfoNCE puede resolverse reconociendo al sujeto (amplitud, colocación de electrodos, ritmo propio) en lugar de la actividad, y que eso perjudica la generalización a personas nuevas; el modo `mismo_sujeto` debería reducir esa información.

In [ ]:
def load_cpc(kind, hname, batch_mode, seed):
    path = RUN_DIR / "cpc" / f"{run_name(kind, hname, batch_mode, seed)}.pt"
    if not path.exists():
        return None
    model = build_model(kind, HORIZONS[hname], seed)
    model.load_state_dict(torch.load(path, map_location=DEVICE)["state_dict"])
    return model.eval()

best_model = load_cpc(BEST_KIND, BEST_H, BEST_BATCH, SEEDS[0])
random_model = build_model(BEST_KIND, HORIZONS[BEST_H], SEEDS[0]).eval()
diag = {}
for name, model in (("CPC", best_model), ("sin entrenar", random_model)):
    ev, preds = probe_eval(model, (BEST_REP,), {"validación": VAL}, return_preds=True)
    diag[name] = {"metrics": ev[BEST_REP]["validación"], "pred": preds[BEST_REP]["validación"]}

display(pd.DataFrame({name: d["metrics"]["f1_per_class"] + [d["metrics"]["f1_macro"]] for name, d in diag.items()},
                     index=LABELS + ["F1 macro"]).T.round(3))

fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
for ax, (name, d) in zip(axes[:2], diag.items()):
    cm = np.array(d["metrics"]["confusion"])
    ax.imshow(cm, cmap="Blues", vmin=0, vmax=1)
    ax.set(xticks=range(len(LABELS)), xticklabels=LABELS, yticks=range(len(LABELS)), yticklabels=LABELS,
           xlabel="predicción", ylabel="real", title=f"{name}: {BEST_KIND} · {BEST_REP} (validación)")
    ax.grid(False)
    for i in range(len(LABELS)):
        for j in range(len(LABELS)):
            ax.text(j, i, f"{cm[i, j]:.2f}", ha="center", va="center", fontsize=7, color="white" if cm[i, j] > 0.55 else "black")

edges_s = [0, 0.5, 1, 2, 4, np.inf]
since = VAL["frame"]["steps_in_segment"].to_numpy() / TARGET_FS
bucket = pd.cut(since, edges_s, right=False, labels=["0–0.5 s", "0.5–1 s", "1–2 s", "2–4 s", "≥ 4 s"])
for name, d in diag.items():
    f1_b = [f1_score(VAL["y"][bucket == b], d["pred"][bucket == b], average="macro") if (bucket == b).any() else np.nan
            for b in bucket.categories]
    axes[2].plot(bucket.categories, f1_b, marker="o", label=name)
axes[2].set(title="F1 macro según el tiempo desde el inicio de la actividad", ylabel="F1 macro (validación)",
            xlabel="tiempo transcurrido de la actividad etiquetada")
axes[2].legend(frameon=False)
plt.tight_layout()
plt.show()
print("Ventanas por tramo:", pd.Series(bucket).value_counts().reindex(bucket.categories).to_dict())

In [ ]:
# Orden del protocolo: ¿cuánto de la etiqueta se predice sólo con la actividad anterior?
def previous_activity(frame):
    # Última etiqueta anotada antes del segmento que termina en end_step (-1 = inicio del ensayo)
    prev = np.full(len(frame), -1, dtype=np.int64)
    cols = zip(frame["trial_idx"].to_numpy(), frame["end_step"].to_numpy(), frame["steps_in_segment"].to_numpy())
    for i, (t, end, n_in) in enumerate(cols):
        before = labels_series[offsets[t]:offsets[t] + end - n_in + 1]
        labeled = np.flatnonzero(before >= 0)
        if len(labeled):
            prev[i] = before[labeled[-1]]
    return prev

prev_train = previous_activity(sup_train)
prev_val = previous_activity(VAL["frame"])
names = {-1: "inicio", **dict(enumerate(LABELS))}
transitions = pd.crosstab(pd.Series(prev_train).map(names).rename("anterior"),
                          pd.Series(sup_train["target"].to_numpy()).map(names).rename("actual"), normalize="index")
oracle_map = pd.crosstab(prev_train, sup_train["target"].to_numpy()).idxmax(axis=1).to_dict()
oracle_pred = np.array([oracle_map.get(p, 0) for p in prev_val])
print("Transiciones en entrenamiento (fila = actividad anterior):")
display(transitions.reindex(columns=LABELS).round(2))

y_val, pred_cpc, pred_rand = VAL["y"], diag["CPC"]["pred"], diag["sin entrenar"]["pred"]
rows = []
for b in bucket.categories:
    m = np.asarray(bucket == b)
    hit = pred_cpc[m] == y_val[m]
    rows.append({"tramo": b, "ventanas": int(m.sum()),
                 "acc CPC": hit.mean(), "acc sin entrenar": (pred_rand[m] == y_val[m]).mean(),
                 "acc oráculo de orden": (oracle_pred[m] == y_val[m]).mean(),
                 "CPC predice la actividad anterior": (pred_cpc[m] == prev_val[m]).mean(),
                 "aciertos CPC que el oráculo también acierta": (hit & (oracle_pred[m] == y_val[m])).sum() / max(hit.sum(), 1)})
order_table = pd.DataFrame(rows).set_index("tramo")
display(order_table.round(3))

early = np.asarray(bucket == bucket.categories[0])
recall = lambda pred: [((pred == y_val) & (y_val == k) & early).sum() / max(((y_val == k) & early).sum(), 1) for k in range(len(LABELS))]
display(pd.DataFrame({"CPC": recall(pred_cpc), "sin entrenar": recall(pred_rand), "oráculo de orden": recall(oracle_pred),
                      "ventanas": [int(((y_val == k) & early).sum()) for k in range(len(LABELS))]},
                     index=pd.Index(LABELS, name=f"recall en {bucket.categories[0]}")).T.round(3))

In [ ]:
# Sonda de sujeto: ¿cuánto se puede reconocer a la persona desde c_T?
probe_frame = PROBE_TRAIN["frame"]
half = (probe_frame["trial_idx"] % 2 == 0).to_numpy()            # mitad de los ensayos de cada sujeto
subj = probe_frame["subject"].to_numpy()
candidates = {"sin entrenar": random_model}
for batch_mode in BATCH_MODES:
    m = load_cpc(BEST_KIND, BEST_H, batch_mode, SEEDS[0])
    if m is not None:
        candidates[f"CPC {batch_mode}"] = m
rows = []
for name, model in candidates.items():
    X = extract_reps(model, PROBE_TRAIN["starts"], ("c_T",))["c_T"]
    clf = fit_probe(X[half], subj[half])
    pred = clf.predict(X[~half])
    rows.append({"representación": name, "accuracy de sujeto": accuracy_score(subj[~half], pred),
                 "balanced accuracy": balanced_accuracy_score(subj[~half], pred)})
subject_probe = pd.DataFrame(rows).set_index("representación")
print(f"{BEST_KIND}, horizonte {BEST_H} | azar ≈ {1 / len(np.unique(subj)):.2f}")
display(subject_probe.round(3))

# 9. Línea base tabular y vista híbrida

Las 202 características manuales del Avance 2 (en `%MVC`) con la misma sonda (`StandardScaler` + regresión logística balanceada) y las mismas ventanas de entrenamiento y validación. Es la referencia que la representación aprendida debe superar. Aquí no se aplica la selección por información mutua del Avance 2, para que las cuatro vistas se comparen con el mismo clasificador.

**Vista híbrida:** la mejor representación CPC concatenada con (a) las características prioritarias del Avance 2 —`tail_mean`, `slope`, `tail_std` y `mean_abs_slope` de cada canal— o (b) todas las características.

In [ ]:
features = pd.read_csv(EXPORT_DIR / "features.csv.gz")
FEATURE_COLUMNS = meta["feature_columns"]
PRIORITY_COLUMNS = [c for c in FEATURE_COLUMNS if c.split("_", 1)[1] in ("tail_mean", "slope", "tail_std", "mean_abs_slope")]
key = ["trial_idx", "end_step"]
feat_lookup = features.set_index(key)[FEATURE_COLUMNS]

def features_for(es, columns):
    return feat_lookup.loc[list(zip(es["frame"]["trial_idx"], es["frame"]["end_step"])), columns].to_numpy(dtype=np.float32)

F_train_all, F_val_all = features_for(PROBE_TRAIN, FEATURE_COLUMNS), features_for(VAL, FEATURE_COLUMNS)
F_train_pri, F_val_pri = features_for(PROBE_TRAIN, PRIORITY_COLUMNS), features_for(VAL, PRIORITY_COLUMNS)
R_train = extract_reps(best_model, PROBE_TRAIN["starts"], (BEST_REP,))[BEST_REP]
R_val = extract_reps(best_model, VAL["starts"], (BEST_REP,))[BEST_REP]

views = {
    "tabular (202 características)": (F_train_all, F_val_all),
    "tabular (prioritarias)": (F_train_pri, F_val_pri),
    f"CPC {BEST_KIND} · {BEST_REP}": (R_train, R_val),
    "híbrido (CPC + prioritarias)": (np.hstack([R_train, F_train_pri]), np.hstack([R_val, F_val_pri])),
    "híbrido (CPC + 202)": (np.hstack([R_train, F_train_all]), np.hstack([R_val, F_val_all])),
}
view_results = {}
for name, (Xtr, Xva) in views.items():
    view_results[name] = metrics(VAL["y"], fit_probe(Xtr, PROBE_TRAIN["y"]).predict(Xva))
display(pd.DataFrame({n: m["f1_per_class"] + [m["f1_macro"], m["balanced_accuracy"]] for n, m in view_results.items()},
                     index=LABELS + ["F1 macro", "balanced acc"]).T.round(3))

# 10. Etapa 3 — ajuste fino

La sonda lineal mide qué tan buena es la representación congelada, pero no es el mejor clasificador posible. Aquí se entrena el modelo completo (encoder + `g_ar` + cabeza lineal sobre la representación) con las etiquetas, para las `FINETUNE_TOP_K` mejores configuraciones de la etapa 2 (cada una con su mejor representación) y con todas las `FINETUNE_SEEDS`:

- **`preentrenado`:** inicializado con el checkpoint de CPC de la misma semilla; tasa de aprendizaje menor en el *backbone* (`FINETUNE_LR_BACKBONE`) que en la cabeza.
- **`desde_cero`:** la misma arquitectura con pesos aleatorios y las mismas tasas. Es el control que dice si CPC aporta cuando también se entrena con etiquetas.

Entropía cruzada ponderada por clase (inverso de la frecuencia en entrenamiento), todas las ventanas supervisadas de entrenamiento (salto de 0.5 s) y parada temprana por F1 macro de validación.

In [ ]:
class CPCClassifier(nn.Module):
    def __init__(self, cpc_model, representation, n_classes=len(LABELS)):
        super().__init__()
        self.encoder, self.ar, self.representation = cpc_model.encoder, cpc_model.ar, representation
        dim = cpc_model.cfg.c_dim * (2 if representation == "c_T+media" else 1)
        self.head = nn.Sequential(nn.LayerNorm(dim), nn.Dropout(0.2), nn.Linear(dim, n_classes))

    def forward(self, x):
        z = self.encoder(x).transpose(1, 2)
        return self.head(summarize(self.ar(z), self.representation))


@torch.no_grad()
def predict(clf_model, starts, batch=512):
    clf_model.eval()
    return np.concatenate([clf_model(get_windows(starts[s:s + batch])).argmax(1).cpu().numpy()
                           for s in range(0, len(starts), batch)])


FT_TRAIN = eval_set(sup_train, FINETUNE_TRAIN_CAP)
class_counts = np.bincount(FT_TRAIN["y"], minlength=len(LABELS))
CLASS_WEIGHTS = torch.tensor(class_counts.sum() / (len(LABELS) * class_counts), dtype=torch.float32, device=DEVICE)


def finetune(config, variant, seed):
    kind, hname, batch_mode, rep = config
    path = RUN_DIR / "finetune" / f"{kind}_{hname}_{batch_mode}_{rep}_{variant}_s{seed}.json"
    if path.exists() and not RERUN:
        r = json.loads(path.read_text())
        print(f"{path.stem}: corrida guardada | mejor época {r['best_epoch']} de {len(r['history'])} | "
              f"{r['minutes']:.0f} min | F1 val {r['eval']['validación']['f1_macro']:.3f}")
        return
    base = load_cpc(kind, hname, batch_mode, seed) if variant == "preentrenado" else build_model(kind, HORIZONS[hname], seed)
    if base is None:                                    # no hay checkpoint de esa semilla: se usa el de SEEDS[0]
        base = load_cpc(kind, hname, batch_mode, SEEDS[0])
    torch.manual_seed(seed)
    model = CPCClassifier(base, rep).to(DEVICE)
    opt = torch.optim.AdamW([
        {"params": list(model.encoder.parameters()) + list(model.ar.parameters()), "lr": FINETUNE_LR_BACKBONE},
        {"params": model.head.parameters(), "lr": FINETUNE_LR_HEAD},
    ], weight_decay=1e-4)
    rng = np.random.default_rng(seed)
    y_all = torch.as_tensor(np.array(FT_TRAIN["y"]), device=DEVICE)
    best, bad, history, t0 = {"f1": -1.0, "epoch": 0, "state": None}, 0, [], time.time()
    for epoch in range(1, FINETUNE_EPOCHS + 1):
        model.train()
        order = rng.permutation(len(FT_TRAIN["y"]))
        run_loss = 0.0
        for s in range(0, len(order), FINETUNE_BATCH):
            idx = order[s:s + FINETUNE_BATCH]
            loss = F.cross_entropy(model(get_windows(FT_TRAIN["starts"][idx])), y_all[idx], weight=CLASS_WEIGHTS)
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()
            run_loss += loss.item() * len(idx)
        f1_val = f1_score(VAL["y"], predict(model, VAL["starts"]), average="macro")
        history.append({"epoch": epoch, "train_loss": run_loss / len(order), "f1_val": f1_val})
        mark = ""
        if f1_val > best["f1"] + 1e-4:
            best, bad, mark = {"f1": f1_val, "epoch": epoch, "state": copy.deepcopy(model.state_dict())}, 0, " *"
        else:
            bad += 1
        print(f"   {variant} s{seed} | época {epoch:2d} | loss {run_loss / len(order):.3f} | F1 val {f1_val:.3f}{mark}")
        if bad >= FINETUNE_PATIENCE:
            break
    model.load_state_dict(best["state"])
    result = {"variant": variant, "seed": seed, "model": kind, "horizons_name": hname, "batch_mode": batch_mode,
              "rep": rep, "best_epoch": best["epoch"], "history": history, "minutes": (time.time() - t0) / 60,
              "eval": {"validación": metrics(VAL["y"], predict(model, VAL["starts"]))}}
    torch.save({"state_dict": model.state_dict(), "result": result, "preprocessing": prep}, path.with_suffix(".pt"))
    path.write_text(json.dumps(result))
    del model, base


best_rep_per_config = summary2["mean"].groupby(level=["modelo", "horizonte", "batch"]).idxmax()
FT_CONFIGS = list(summary2.loc[list(best_rep_per_config), "mean"].sort_values(ascending=False).index[:FINETUNE_TOP_K])
print("Configuraciones para ajuste fino:", FT_CONFIGS)
for config in FT_CONFIGS:
    for seed in FINETUNE_SEEDS:
        for variant in ("preentrenado", "desde_cero"):
            finetune(config, variant, seed)

ft_rows = []
for path in sorted((RUN_DIR / "finetune").glob("*.json")):
    r = json.loads(path.read_text())
    m = r["eval"]["validación"]
    ft_rows.append({"modelo": r["model"], "horizonte": r["horizons_name"], "batch": r["batch_mode"], "rep": r["rep"],
                    "variante": r["variant"], "semilla": r["seed"], "mejor época": r["best_epoch"], "min": r["minutes"],
                    "F1": m["f1_macro"], "bal_acc": m["balanced_accuracy"], **{l: f for l, f in zip(LABELS, m["f1_per_class"])}})
finetune_results = pd.DataFrame(ft_rows)
ft_keys = ["modelo", "horizonte", "batch", "rep", "variante"]
display(finetune_results.groupby(ft_keys)[["F1", "bal_acc", *LABELS]].mean().round(3)
        .join(finetune_results.groupby(ft_keys)["F1"].agg(["std", "count"]).round(3)))

# 11. Comparación final en validación

Todas las vistas evaluadas con los sujetos 11–12, primero en conjunto y después por sujeto. La decisión de qué llevar a prueba se toma **aquí**, con validación; la sección 12 sólo mide una vez lo ya decidido.

In [ ]:
final_rows = []
r_rand = rand_all.loc[(BEST_KIND, BEST_REP)]
final_rows.append({"vista": f"{BEST_KIND} sin entrenar · {BEST_REP} (sonda)", "F1 media": r_rand["mean"],
                   "F1 desv.": r_rand["std"], "n": r_rand["count"]})
r_best = summary2.loc[BEST]
final_rows.append({"vista": f"{BEST_KIND} CPC {BEST_H} · {BEST_BATCH} · {BEST_REP} (sonda)", "F1 media": r_best["mean"],
                   "F1 desv.": r_best["std"], "n": r_best["count"]})
for name, m in view_results.items():
    if name.startswith(("tabular", "híbrido")):
        final_rows.append({"vista": name + " (sonda)", "F1 media": m["f1_macro"], "F1 desv.": np.nan, "n": 1})
for (kind, hname, batch_mode, rep, variant), g in finetune_results.groupby(ft_keys):
    final_rows.append({"vista": f"ajuste fino {variant} · {kind} {hname} · {batch_mode} · {rep}", "F1 media": g["F1"].mean(),
                       "F1 desv.": g["F1"].std() if len(g) > 1 else np.nan, "n": len(g)})
final_table = pd.DataFrame(final_rows).set_index("vista").sort_values("F1 media", ascending=False)
display(final_table.round(3))

fig, ax = plt.subplots(figsize=(10, 0.45 * len(final_table) + 1))
ax.barh(final_table.index[::-1], final_table["F1 media"][::-1], xerr=final_table["F1 desv."][::-1].fillna(0), color="#2a78d6")
for i, v in enumerate(final_table["F1 media"][::-1]):
    ax.text(v + 0.005, i, f"{v:.3f}", va="center", fontsize=8)
ax.set(xlabel="F1 macro (validación, sujetos 11–12)", title=f"Comparación final — {TARGET_FS} Hz")
plt.tight_layout()
plt.show()

### F1 por sujeto de validación

El promedio de validación mezcla a dos personas. La tabla separa el F1 de cada sujeto (media y desviación entre semillas en el ajuste fino). Si una diferencia entre vistas aparece en un sujeto y no en el otro, o es menor que la diferencia entre sujetos, no conviene darla por establecida: con dos personas en validación no se puede estimar cuánto variaría con otras.

In [ ]:
subjects_val = VAL["frame"]["subject"].to_numpy()
subject_ids = np.unique(subjects_val)

def f1_by_subject(pred):
    out = {f"sujeto {s}": f1_score(VAL["y"][subjects_val == s], pred[subjects_val == s], average="macro") for s in subject_ids}
    out["total"] = f1_score(VAL["y"], pred, average="macro")
    return out

rows = []
for name in ("tabular (202 características)", f"CPC {BEST_KIND} · {BEST_REP}", "híbrido (CPC + prioritarias)"):
    Xtr, Xva = views[name]
    rows.append({"vista": f"{name} (sonda)", **f1_by_subject(fit_probe(Xtr, PROBE_TRAIN["y"]).predict(Xva))})
for path in sorted((RUN_DIR / "finetune").glob("*.pt")):
    ckpt = torch.load(path, map_location=DEVICE)
    r = ckpt["result"]
    model = CPCClassifier(build_model(r["model"], HORIZONS[r["horizons_name"]], r["seed"]), r["rep"]).to(DEVICE)
    model.load_state_dict(ckpt["state_dict"])
    rows.append({"vista": f"ajuste fino {r['variant']} · {r['model']} {r['horizons_name']} · {r['rep']}",
                 **f1_by_subject(predict(model, VAL["starts"]))})
    del model
by_subject = pd.DataFrame(rows)
cols = [f"sujeto {s}" for s in subject_ids] + ["total"]
summary_subject = by_subject.groupby("vista")[cols].agg(["mean", "std"]).sort_values(("total", "mean"), ascending=False)
display(summary_subject.round(3))

# 12. Prueba (sujetos 13–14) — una sola vez

Poner `TEST_OPEN = True` sólo cuando todas las decisiones estén congeladas (frecuencia, modelo, horizonte, modo de batch, representación y vista). La celda evalúa la mejor vista con sonda, la tabular, la híbrida con prioritarias y los modelos con ajuste fino, y guarda el resultado en `test_resultados.json`.

In [ ]:
if not TEST_OPEN:
    print("Prueba cerrada (TEST_OPEN = False).")
else:
    test_rows = {}
    R_test = extract_reps(best_model, TEST["starts"], (BEST_REP,))[BEST_REP]
    F_test_all, F_test_pri = features_for(TEST, FEATURE_COLUMNS), features_for(TEST, PRIORITY_COLUMNS)
    test_views = {
        f"CPC {BEST_KIND} · {BEST_REP} (sonda)": (R_train, R_test),
        "tabular (202 características)": (F_train_all, F_test_all),
        "híbrido (CPC + prioritarias)": (np.hstack([R_train, F_train_pri]), np.hstack([R_test, F_test_pri])),
    }
    for name, (Xtr, Xte) in test_views.items():
        test_rows[name] = metrics(TEST["y"], fit_probe(Xtr, PROBE_TRAIN["y"]).predict(Xte))
    for path in sorted((RUN_DIR / "finetune").glob("*.pt")):
        ckpt = torch.load(path, map_location=DEVICE)
        r = ckpt["result"]
        model = CPCClassifier(build_model(r["model"], HORIZONS[r["horizons_name"]], r["seed"]), r["rep"]).to(DEVICE)
        model.load_state_dict(ckpt["state_dict"])
        test_rows[f"ajuste fino {r['variant']} s{r['seed']}"] = metrics(TEST["y"], predict(model, TEST["starts"]))
    (RUN_DIR / "test_resultados.json").write_text(json.dumps(test_rows))
    display(pd.DataFrame({n: m["f1_per_class"] + [m["f1_macro"], m["balanced_accuracy"]] for n, m in test_rows.items()},
                         index=LABELS + ["F1 macro", "balanced acc"]).T.round(3))

# 13. Cómo leer los resultados

- **¿Aporta CPC?** Columnas de ganancia de la etapa 1 (CPC menos la misma red sin entrenar) y la comparación de la etapa 2 con las mismas semillas. En el ajuste fino, comparar `preentrenado` con `desde_cero`: si empatan, la ventaja está en la arquitectura y en las etiquetas, no en el preentrenamiento.
- **¿Funciona `mismo_sujeto`?** Comparar las filas `mezclado` y `mismo_sujeto` del mismo modelo y horizonte, y la sonda de sujeto de la sección 8: si `mismo_sujeto` reconoce menos a la persona y clasifica igual o mejor la actividad, ese atajo era parte del problema.
- **Representación:** con etiqueta en el instante final se espera que `c_T` o `cola` superen a `media`. Si `media` gana, el modelo aprovecha el contexto previo, lo que hay que contrastar con el análisis del orden del protocolo.
- **Orden del protocolo:** si el modelo acierta casi tanto como el oráculo de orden en el primer medio segundo de cada actividad, o acierta clases que el oráculo no puede predecir (como `LT`), se puede saber si usa la secuencia o la señal. Lo primero no se transfiere a un uso real.
- **Tamaño de las diferencias:** con una sola semilla o dos sujetos de validación, diferencias de ±0.02–0.03 de F1 no son concluyentes. Revisar siempre la desviación entre semillas y el F1 por sujeto (sección 11).
- **Líneas de referencia:** la vista con características manuales y la red sin entrenar. Como referencia externa, a 50 Hz la línea base tabular del Avance 2 obtuvo 0.420 de F1 en validación con selección por información mutua.
- **Comparar frecuencias:** ejecutar el notebook con otro `TARGET_FS` y comparar las tablas de validación de cada ejecución. La frecuencia se elige con validación, no con prueba.
- **Límite conocido:** `LF`/`LT` y `PT`/`PF` difieren por la altura a la que se manipula la carga, información de postura que la sEMG casi no contiene. Esas confusiones son un techo de la señal, no del pipeline.